# Regresion de precio de estudios usando CRISP-DM

Esta libreta documenta una solucion de **regresion supervisada**
para estimar el precio normal de un estudio.


## 1. Comprension del negocio

- Solucion: regresion supervisada con `Ridge`.
- Objetivo: estimar el **precio normal** de un estudio.
- Unidad de analisis: estudio individual activo.


## 2. Comprension de los datos

Primero se carga el dataset producido por ETL y se revisa su
estructura general.


In [1]:
import os
import json
import matplotlib.pyplot as plt
import pickle
import unicodedata
from pathlib import Path

os.environ["LOKY_MAX_CPU_COUNT"] = "1"

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

def make_onehot_encoder():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)

def normalize_text(value, fallback):
    if pd.isna(value):
        return fallback
    text = str(value).strip().lower()
    text = "".join(
        character
        for character in unicodedata.normalize("NFD", text)
        if unicodedata.category(character) != "Mn"
    )
    text = " ".join(text.split())
    return text or fallback

BACKEND = next(
    path for path in [Path.cwd(), *Path.cwd().parents]
    if (path / "package.json").exists() and (path / "src").exists()
)
RANDOM_STATE = 42


Librerias cargadas correctamente.
Semilla fija: 42


In [2]:
df = pd.read_csv(BACKEND / "05_Datasets" / "02_regresion_estudios_dataset.csv")
print(df.head().to_string(index=False))


study_id study_code                     study_name  is_synthetic  type             method  parameter_count  duration_minutes sample_type requires_special_processing  normal_price
       1    GLU-001                        GLUCOSA         False study         sin_metodo                0                60     unknown                         NaN         110.0
       2       BH01             BIOMETRIA HEMATICA         False study       AUTOMATIZADO                0                30     unknown                         NaN         180.0
       3       QS06  QUIMICA SANGUINEA 6 ELEMENTOS         False study ESPECTROFOTOMETRIA                0                45     unknown                         NaN         220.0
       4       QS12 QUIMICA SANGUINEA 12 ELEMENTOS         False study ESPECTROFOTOMETRIA                0                60     unknown                         NaN         350.0
       5      GLU01                        GLUCOSA         False study         ENZIMATICO                

In [3]:
print(df.shape)


(2000, 11)


In [4]:
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 11 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   study_id                     2000 non-null   int64  
 1   study_code                   2000 non-null   str    
 2   study_name                   2000 non-null   str    
 3   is_synthetic                 2000 non-null   bool   
 4   type                         2000 non-null   str    
 5   method                       2000 non-null   str    
 6   parameter_count              2000 non-null   int64  
 7   duration_minutes             2000 non-null   int64  
 8   sample_type                  2000 non-null   str    
 9   requires_special_processing  1958 non-null   object 
 10  normal_price                 2000 non-null   float64
dtypes: bool(1), float64(1), int64(3), object(1), str(5)
memory usage: 158.3+ KB


In [5]:
print(df[["parameter_count", "duration_minutes", "normal_price"]].describe().round(2).to_string())


       parameter_count  duration_minutes  normal_price
count          2000.00           2000.00       2000.00
mean             15.14            213.83        600.32
std               8.91            210.03        230.39
min               0.00             15.00         80.00
25%               7.00            105.00        420.00
50%              15.00            150.00        600.00
75%              23.00            210.00        770.00
max              32.00           2880.00       1260.00


In [6]:
print(
    df[
        [
            "parameter_count",
            "duration_minutes",
            "method",
            "sample_type",
            "requires_special_processing",
            "normal_price",
        ]
    ].isnull().sum().to_string()
)


parameter_count                 0
duration_minutes                0
method                          0
sample_type                     0
requires_special_processing    42
normal_price                    0


### Control de calidad inicial

Las salidas anteriores dejan evidencia de:

- muestra inicial
- dimensiones del dataset
- tipos de dato
- estadisticos descriptivos
- valores faltantes

En la preparacion tambien se eliminan duplicados y registros no
utilizables.


### Descripcion de las variables

Las variables principales de esta propuesta son:

- `parameter_count`
- `duration_minutes`
- `method`
- `sample_type`
- `requires_special_processing`
- `normal_price` como variable objetivo


## 3. Preparacion de los datos

Se limpian nulos, se normalizan variables categoricas y se separan
las variables X y la variable Y.


In [7]:
work = df.copy()
work["parameter_count"] = pd.to_numeric(work["parameter_count"], errors="coerce")
work["duration_minutes"] = pd.to_numeric(work["duration_minutes"], errors="coerce")
work["normal_price"] = pd.to_numeric(work["normal_price"], errors="coerce")
work["is_synthetic"] = work["is_synthetic"].astype(str).str.lower().isin(["true", "1"])
work["type"] = work["type"].astype(str).str.lower().str.strip()

work = work.drop_duplicates(subset=["study_id"], keep="last")
work = work.dropna(subset=["study_id", "normal_price"])
work = work[
    (work["type"] == "study")
    & (work["normal_price"] > 0)
    & (
        work["parameter_count"].isna()
        | ((work["parameter_count"] >= 0) & np.isfinite(work["parameter_count"]))
    )
    & (
        work["duration_minutes"].isna()
        | ((work["duration_minutes"] > 0) & np.isfinite(work["duration_minutes"]))
    )
].copy()

work["method"] = work["method"].map(lambda value: normalize_text(value, "sin_metodo"))
work["sample_type"] = work["sample_type"].map(lambda value: normalize_text(value, "unknown"))
work["requires_special_processing"] = work["requires_special_processing"].map(
    lambda value: "true"
    if str(value).strip().lower() in {"true", "1", "yes", "si"}
    else "false"
    if str(value).strip().lower() in {"false", "0", "no"}
    else "sin_especificar"
)

features = [
    "parameter_count",
    "duration_minutes",
    "method",
    "sample_type",
    "requires_special_processing",
]
target = "normal_price"

X = work[features]
y = work[target]
print("Filas utiles:", len(work))
print("Variables X:", features)


Filas utiles: 2000
Variables X: ['parameter_count', 'duration_minutes', 'method', 'sample_type', 'requires_special_processing']


## 4. Modelado y entrenamiento

Se construye un `Pipeline` con:

- imputacion de nulos
- escalado para variables numericas
- one-hot encoding para variables categoricas
- modelo `Ridge`

Para reproducibilidad se usa `RANDOM_STATE = 42`.


In [8]:
numeric_features = ["parameter_count", "duration_minutes"]
categorical_features = ["method", "sample_type", "requires_special_processing"]

stratify = (
    work["is_synthetic"]
    if work["is_synthetic"].nunique() > 1 and work["is_synthetic"].value_counts().min() >= 2
    else None
)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    shuffle=True,
    stratify=stratify,
)

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="median")),
                    ("scaler", StandardScaler()),
                ]
            ),
            numeric_features,
        ),
        (
            "categorical",
            Pipeline(
                steps=[
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    ("onehot", make_onehot_encoder()),
                ]
            ),
            categorical_features,
        ),
    ]
)

model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", Ridge(alpha=0.10)),
    ]
)

model.fit(X_train, y_train)
print("Modelo entrenado correctamente.")


Modelo entrenado correctamente.


## 5. Evaluacion

Se comparan valores reales contra valores predichos y se calculan
las metricas principales: MAE, RMSE y R2.


In [9]:
train_pred = model.predict(X_train)
test_pred = model.predict(X_test)

metrics = {
    "train": {
        "mae": round(float(mean_absolute_error(y_train, train_pred)), 4),
        "rmse": round(float(np.sqrt(mean_squared_error(y_train, train_pred))), 4),
        "r2": round(float(r2_score(y_train, train_pred)), 4),
    },
    "test": {
        "mae": round(float(mean_absolute_error(y_test, test_pred)), 4),
        "rmse": round(float(np.sqrt(mean_squared_error(y_test, test_pred))), 4),
        "r2": round(float(r2_score(y_test, test_pred)), 4),
    },
}

print(json.dumps(metrics, indent=2, ensure_ascii=False))


{
  "train": {
    "mae": 27.222,
    "rmse": 36.5651,
    "r2": 0.9748
  },
  "test": {
    "mae": 30.404,
    "rmse": 41.4118,
    "r2": 0.9674
  }
}


### Interpretacion de la evaluacion

Un R2 alto y errores bajos indican que el modelo reproduce bien la
variacion del precio normal. La siguiente salida compara ejemplos
reales contra ejemplos predichos.


In [10]:
predicciones = X_test.copy()
predicciones["normal_price"] = y_test.values
predicciones["predicted_price"] = np.round(test_pred, 2)
predicciones["residual"] = np.round(predicciones["normal_price"] - predicciones["predicted_price"], 2)

print(
    predicciones[
        ["normal_price", "predicted_price", "residual"]
    ].head(8).to_string(index=False)
)


study_id      study_code  normal_price  predicted_price  residual
     300 ECN-CAT-HEM-031         510.0           500.72      9.28
     808 ECN-CAT-END-094         270.0           283.28    -13.28
     239 ECN-CAT-COA-023         390.0           409.77    -19.77
     761 ECN-CAT-URO-088         210.0           243.31    -33.31
    1662 ECN-CAT-END-134         580.0           562.72     17.28
    1821 ECN-CAT-URO-174         390.0           333.57     56.43
     561 ECN-CAT-URO-063         490.0           481.80      8.20
    1054 ECN-CAT-MIC-125         570.0           561.20      8.80


### Grafica de ajuste

La siguiente grafica permite comparar visualmente los valores
reales contra los valores predichos por el modelo.


In [11]:
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(
    predicciones["normal_price"],
    predicciones["predicted_price"],
    alpha=0.65,
    color="#1f77b4",
)

min_value = min(
    predicciones["normal_price"].min(),
    predicciones["predicted_price"].min(),
)
max_value = max(
    predicciones["normal_price"].max(),
    predicciones["predicted_price"].max(),
)
ax.plot(
    [min_value, max_value],
    [min_value, max_value],
    linestyle="--",
    color="#444444",
    linewidth=1.5,
)
ax.set_title("Regresion: valor real vs valor predicho")
ax.set_xlabel("Precio real")
ax.set_ylabel("Precio predicho")
ax.grid(alpha=0.25)
plt.tight_layout()
plt.show()


## 6. Exportacion del modelo

El modelo entrenado se guarda en
`07_Modelos/regression_price_model.pkl`.
Ese mismo nombre es el que espera el flujo Python del sistema web
en `ml-artifacts/scripts/regression_train.py`.


In [12]:
model_relative = Path("07_Modelos") / "regression_price_model.pkl"
residuals_relative = Path("05_Datasets") / "02_regresion_reales_predichos_test.csv"
model_path = BACKEND / model_relative
residuals_path = BACKEND / residuals_relative

bundle = {
    "model": model,
    "features": features,
    "numeric_features": numeric_features,
    "categorical_features": categorical_features,
}

with model_path.open("wb") as file:
    pickle.dump(bundle, file)

predicciones.to_csv(residuals_path, index=False)

print("Modelo exportado en:", model_relative.as_posix())
print("Predicciones test guardadas en:", residuals_relative.as_posix())


Modelo exportado en: 07_Modelos/regression_price_model.pkl
Predicciones test guardadas en: 05_Datasets/02_regresion_reales_predichos_test.csv


## 7. Ejemplo de prediccion

Finalmente se realiza una prediccion usando el modelo ya entrenado.


In [13]:
muestra = pd.DataFrame(
    [
        {
            "parameter_count": X.iloc[0]["parameter_count"],
            "duration_minutes": X.iloc[0]["duration_minutes"],
            "method": X.iloc[0]["method"],
            "sample_type": X.iloc[0]["sample_type"],
            "requires_special_processing": X.iloc[0]["requires_special_processing"],
        }
    ]
)

precio_estimado = model.predict(muestra)[0]
print("Entrada:")
print(json.dumps(muestra.iloc[0].to_dict(), indent=2, ensure_ascii=False))
print("\nPrecio estimado:", round(float(precio_estimado), 2))


Entrada:
{
  "parameter_count": 0.0,
  "duration_minutes": 60.0,
  "method": "sin_metodo",
  "sample_type": "unknown",
  "requires_special_processing": "sin_especificar"
}

Precio estimado: 117.5163
